# AMPS I · Week 01 — HW1: Quadrature accuracy
**Due:** before the Week 2 lecture · **Points:** 100 · **Submit:** this notebook with all outputs

Rules as in HW0 (restart & run all; AI-usage log at the end).

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## Q1 — Theory (20 pts)
Let $\{(x_q, w_q)\}_{q=1}^m$ be any $m$-point rule on $[0,1]$.

**(a) (8)** Show that it cannot be exact for all polynomials of degree $2m$.
*(Hint: $p(x)=\prod_q (x-x_q)^2$.)*

**(b) (12)** Let $x_q$ be the roots of the degree-$m$ Legendre polynomial $P_m$ shifted to $[0,1]$, and choose $w_q$
so that the rule is exact on $\mathscr{P}_{m-1}$. Show that it is exact on $\mathscr{P}_{2m-1}$.
*(Hint: divide $p\in\mathscr{P}_{2m-1}$ by $P_m$: $p = sP_m + r$.)*

> *your proofs*

## Q2 — What Basix actually delivers (25 pts)
Exact monomial integrals on the reference cells:
interval $\frac{1}{a+1}$; quadrilateral $\frac{1}{(a+1)(b+1)}$; triangle $\frac{a!\,b!}{(a+b+2)!}$; tetrahedron $\frac{a!\,b!\,c!}{(a+b+c+3)!}$.

**(a) (15)** Write `achieved_degree(cell, k)`: build the default Basix rule for requested degree `k` and return the
**largest** total degree $d$ such that all monomials of degree $\le d$ are integrated to $10^{-12}$.
Print a table for $k=1,\dots,8$ on all four cells: requested $k$, number of points, achieved $d$.

**(b) (10)** Where is $d > k$? Explain using Q1 and the tensor-product structure of the quadrilateral rule.

In [ ]:
%%fenicsx -np 1
import basix
import numpy as np
from math import factorial
from itertools import product

# TODO: exact(cell, alpha) for a multi-index alpha
# TODO: achieved_degree(cell, k)
# TODO: print the table for interval, quadrilateral, triangle, tetrahedron

> *(b) your explanation*

## Q3 — Newton–Cotes vs Gauss on a hard integrand (25 pts)
$\displaystyle I=\int_0^1 \frac{dx}{1+25(2x-1)^2} = \frac{\arctan 5}{5}$ (the Runge function).

**(a) (10)** Ordinary cell: closed Newton–Cotes with $n=3,5,\dots,21$ points (weights from the Vandermonde system). Save the errors.

**(b) (10)** `%%fenicsx` cell: Gauss rules from Basix with the same numbers of points. Save the errors.

**(c) (5)** Plot both errors vs $n$ (semilog) and explain the difference, including what happens to the NC weights.

In [ ]:
import numpy as np
f = lambda x: 1 / (1 + 25 * (2 * x - 1)**2)
I = np.arctan(5) / 5
# TODO: Newton–Cotes errors for n = 3, 5, ..., 21 → np.savez("/tmp/fnx/hw1_nc.npz", n=..., err=..., minw=...)

In [ ]:
%%fenicsx -np 1
import basix
import numpy as np
# TODO: Gauss errors for the same numbers of points → np.savez("/tmp/fnx/hw1_gauss.npz", n=..., err=...)
# (an m-point Basix interval rule is requested with degree 2m-1)

In [ ]:
# TODO: load both files, semilogy plot of error vs number of points

> *(c) your explanation*

## Q4 — Quadrature in a finite element solve: variational crimes (30 pts)
Use the Week 1 model problem (mixed BCs, $u_{ex}=\sin\pi x\cos\pi y$). Integrate the **load** $\int f v$ and/or the
**stiffness** $\int\nabla u\cdot\nabla v$ with a fixed `quadrature_degree` $q$
(`ufl.Measure("dx", domain=msh, metadata={"quadrature_degree": q})`). Compute the error with a high-degree measure.

**(a) (15)** For P1 and P2, and $q_L\in\{0,1,2,3\}$ for the load (stiffness at default degree), report $L^2$ errors and rates for $N=8,16,32,64$ (`-np 2`).

**(b) (5)** Now under-integrate the **P2 stiffness** with $q_a\in\{1,2\}$. What happens with $q_a=1$? Catch the exception or check the solution.

**(c) (10)** Explain both observations: degree of the integrands, which $q$ is needed to keep the optimal rate, and why the
under-integrated P2 stiffness matrix is singular (count the rank of a one-point element stiffness matrix).

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import numpy as np
import ufl
from dolfinx import fem, mesh, default_scalar_type
from dolfinx.fem.petsc import LinearProblem
comm = MPI.COMM_WORLD
# TODO: solve(N, deg, qa=None, qL=None) → L2 error (None = default degree); tables for (a) and (b)

> *(c) your explanation*

## 🤖 AI-usage log

*Required in every assignment and project. Using an LLM is encouraged and never penalized — an empty or vague log is.*

| # | Tool / model | What I asked (key prompt, shortened) | What it got wrong or missed | How I detected it | How I fixed it |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

**Code provenance** — which cells/functions were mostly generated, which did you write or substantially change?

> …

**Verification** — how do you know the final result is correct (MMS rates, benchmark value, conservation check, parallel = serial)?

> …

**Reflection (2–3 sentences)** — what would you do differently when prompting next time?

> …